# Adaptive KV-Cache Compression + Speculative Decoding on Google Colab
**Draft:** `Qwen/Qwen2.5-0.5B` | **Target:** `Qwen/Qwen2.5-7B` (FP16, FlashAttention-2)

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Steps:
1. Upload `AdaptiveKVCache.zip` (the project code)
2. Install dependencies
3. Run the unit tests (tiny mock models, ~1 min)
4. Run the context sweep (1k/4k/8k/16k) for all 4 methods with Qwen2.5-0.5B -> Qwen2.5-7B
5. View and download the results

In [ ]:
# 0. Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 1. Upload AdaptiveKVCache.zip from your laptop and unzip it
import os, zipfile
from google.colab import files

os.chdir('/content')
uploaded = files.upload()  # pick AdaptiveKVCache.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/AdaptiveKVCache')

os.chdir('/content/AdaptiveKVCache')
print('Working dir:', os.getcwd())
print(sorted(os.listdir('.')))

In [ ]:
# 2. Install dependencies (Colab's preinstalled torch is kept)
%pip install -q -r requirements.txt
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 3. Sanity check: unit tests on tiny mock models (no download)
!python -m pytest tests/test_framework.py -v

In [ ]:
# 4. MAIN RUN: context sweep 1k/4k/8k/16k for all 4 methods (Baseline, AdaptKV, Spec-only, Proposed)
# FP16 target + draft (no 4-bit), FlashAttention-2, KV budget 0.25 (pure pruning), 5 docs per length.
# Throughput/TPOT measured with timing_mode OFF; d/v split comes from a separate short timing_mode=ON pass.
# First run downloads ~16 GB of weights to Colab's disk (not your laptop).
!python main.py --mode sweep \
    --draft-model Qwen/Qwen2.5-0.5B \
    --target-model Qwen/Qwen2.5-7B \
    --target-quant fp16 \
    --sweep-lengths 1024,4096,8192,16384 \
    --max-new-tokens 256 \
    --gamma 5 \
    --ppl-dataset wikitext \
    --output-dir results

In [ ]:
# 5. View the context-sweep chart and the per-row results
import json
from IPython.display import Image, display

display(Image(filename='results/context_sweep.png'))

with open('results/context_sweep.json') as f:
    report = json.load(f)
cols = ['name', 'context_length', 'actual_prompt_tokens', 'target_quant', 'attn_implementation',
        'timing_mode', 'decode_tokens_per_sec', 'decode_speedup', 'tpot_ms', 'ttft_ms', 'gamma',
        'acceptance_rate_pct', 'mean_accepted_per_step', 'draft_time_ms', 'verify_time_ms',
        'kv_cache_actual_mb', 'compression_ratio', 'perplexity', 'perplexity_std', 'ppl_check']
for row in report['sweep_results']:
    print({c: row.get(c) for c in cols})

In [ ]:
# 6. Cumulative Optimization Sweep (S0 -> S4 at 1K/8K/16K, g=5, FP16 Base, Auto FA2 + Batch Sweep)
# Requires flash-attn (installed in an earlier cell); the run stops with an error if it is missing.
# Writes to results/stages/ so it does not overwrite the main context sweep in results/.
!python main.py --ppl-dataset wikitext --batch-sweep --output-dir results/stages

In [ ]:
# 7. View cumulative S0 -> S4 comparison chart, summary rows, and batch scaling
import json, os
from IPython.display import Image, display

for img_path in ('results/stages/stages_comparison.png',):
    if os.path.exists(img_path):
        display(Image(filename=img_path))
json_path = 'results/stages/stages_comparison.json'
with open(json_path) as f:
    sweep_report = json.load(f)
print('Crossover / Winning Ranges:', sweep_report.get('crossover_analysis', {}))
for r in sweep_report.get('stage_results', sweep_report.get('sweep_results', [])):
    if r.get('status', 'OK') != 'OK':
        print(f"{r.get('stage_label', r.get('id')):<32} | Ctx={r['context_length']:>5} | STATUS={r['status']}")
        continue
    print(
        f"{r.get('stage_label', r.get('id')):<32} | Ctx={r['context_length']:>5} | Attn={r.get('attn_implementation','-')} | "
        f"Dec {r['decode_tokens_per_sec']:>5.2f}/s (TPOT {r['tpot_ms']:.2f}±{r.get('tpot_ms_std',0):.2f}ms) | "
        f"Acc {r.get('acceptance_rate_pct',0):>4.1f}±{r.get('acceptance_rate_pct_std',0):.1f}% | "
        f"D/V={r.get('draft_time_ms',0):.1f}/{r.get('verify_time_ms',0):.1f}ms | "
        f"KV {r['kv_cache_actual_mb']:.2f}MB | VRAM {r.get('peak_memory_mb',0):.1f}MB | "
        f"PPL {r['perplexity']:.2f}±{r.get('perplexity_std',0):.2f}"
    )
if sweep_report.get('batch_size_results'):
    print('\n--- Batch Size Scaling ---')
    for br in sweep_report['batch_size_results']:
        print(br)

### Real-task benchmarks: LongBench + RULER (quality and speed on real prompts)
The sweep above uses synthetic prompts. These cells run **LongBench** (real QA, summarization, few-shot and code tasks with the official prompts and metrics) and **RULER** (needle retrieval, variable tracking and word aggregation at exact 4K/8K/16K lengths) through all 4 methods. Each generation records its task score and its decode speed, acceptance rate and KV memory.

* Uses the **Instruct** pair (task prompts need instruction following). Target and draft are both Instruct, never mixed.
* Results are written after every sample to `results/tasks/task_predictions.jsonl`. If Colab disconnects, re-run the cell and it **resumes** where it stopped.
* About 2-3 h on an A100 with the default 20 samples per task. Lower `--task-samples` for a quick pass.


In [ ]:
# 7b. LongBench (8 tasks, all 6 categories) + RULER (8 tasks x 4K/8K/16K), all 4 methods
!python main.py --mode tasks     --target-model Qwen/Qwen2.5-7B-Instruct     --draft-model Qwen/Qwen2.5-0.5B-Instruct     --target-quant fp16     --suite longbench,ruler     --ruler-lengths 4096,8192,16384     --task-samples 20     --gamma 5     --output-dir results/tasks


In [ ]:
# 7c. View real-task results (tables + charts)
import os
from IPython.display import Image, Markdown, display

display(Markdown(open('results/tasks/task_summary.md', encoding='utf-8').read()))
for img in ('tasks_scores.png', 'tasks_ruler_length.png', 'tasks_efficiency.png'):
    p = os.path.join('results/tasks', img)
    if os.path.exists(p):
        display(Image(filename=p))


In [ ]:
# 8. Download the results folder to your laptop
!zip -qr results.zip results
files.download('results.zip')

### Optional: single-prompt generation

In [ ]:
!python main.py --mode generate \
    --prompt "Explain how speculative decoding accelerates LLM inference:"